In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install torchmetrics

In [ ]:
!pip install pytorch_lightning

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
import os
import json
from transformers import AdamW, get_linear_schedule_with_warmup
from torchmetrics.functional import auroc
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
import pytorch_lightning as pl
from torchmetrics import AUROC, Accuracy, F1Score
from tqdm.auto import tqdm

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
device


device(type='cuda')

In [ ]:
def set_seed(seed = 1234):
    '''Sets the seed of the entire notebook so results are the same every time we run.
    This is for REPRODUCIBILITY.'''
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    # Set a fixed value for the hash seed
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed()

In [ ]:
# Note: worse results after use of this function
def preprocessing(df):
    # Define positive and negative emotions
    positive_emotions = [
        'admiration', 'amusement', 'approval', 'caring', 'desire', 'excitement',
        'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief'
    ]
    negative_emotions = [
        'anger', 'annoyance', 'disappointment', 'disapproval', 'disgust',
        'embarrassment', 'fear', 'grief', 'nervousness', 'remorse', 'sadness'
    ]

    # Function to calculate emotion balance
    def emotion_balance(row):
        positive_score = sum(row[positive_emotions])
        negative_score = sum(row[negative_emotions])
        return positive_score - negative_score

    # Add a column with the emotion balance
    df['emotion_balance'] = df.apply(emotion_balance, axis=1)

    # Sort the dataframe
    df_sorted = df.sort_values(by=['emotion_balance', 'text'], ascending=[False, True])

    # Drop the temporary emotion_balance column
    df_sorted = df_sorted.drop('emotion_balance', axis=1)

    df_sorted = df_sorted[df_sorted['example_very_unclear'] != True].reset_index(drop=True)


    return df_sorted

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, data: pd.DataFrame, tokenizer: BertTokenizer, label_names, max_token_len: int = 128):
        self.tokenizer = tokenizer
        self.data = data
        self.max_token_len = max_token_len
        self.LABEL_COLUMNS = label_names

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index: int):
        data_row = self.data.iloc[index]
        comment_text = data_row.text
        labels = data_row[self.LABEL_COLUMNS]
        encoding = self.tokenizer.encode_plus(
          comment_text,
          add_special_tokens=True,
          max_length=self.max_token_len,
          return_token_type_ids=False,
          padding="max_length",
          truncation=True,
          return_attention_mask=True,
          return_tensors='pt',
        )
        return dict(
          comment_text=comment_text,
          input_ids=encoding["input_ids"].flatten(),
          attention_mask=encoding["attention_mask"].flatten(),
          labels=torch.FloatTensor(labels.values.astype(np.float32))
        )


In [ ]:
class CustomDataModule(pl.LightningDataModule):
    def __init__(self, train_df, test_df, tokenizer, label_names, batch_size=8, max_token_len=128):
        super().__init__()
        self.batch_size = batch_size
        self.train_df = train_df
        self.test_df = test_df
        self.tokenizer = tokenizer
        self.max_token_len = max_token_len
        self.label_names = label_names

    def setup(self, stage=None):
        self.train_dataset = CustomDataset(
            self.train_df,
            self.tokenizer,
            self.label_names,
            self.max_token_len
        )
        self.test_dataset = CustomDataset(
            self.test_df,
            self.tokenizer,
            self.label_names,
            self.max_token_len
        )

    def train_dataloader(self):
        return DataLoader(
          self.train_dataset,
          batch_size=self.batch_size,
          shuffle=True,
          num_workers=2
        )

    def val_dataloader(self):
        return DataLoader(
          self.test_dataset,
          batch_size=self.batch_size,
          num_workers=2
        )

    def test_dataloader(self):
        return DataLoader(
          self.test_dataset,
          batch_size=self.batch_size,
          num_workers=2
        )

In [ ]:
class CustomBertModel(pl.LightningModule):
    def __init__(self, n_classes: int, labels, n_training_steps=None, n_warmup_steps=None):
        super().__init__()
        self.bert = BertModel.from_pretrained(BERT_MODEL_NAME, return_dict=True)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_classes)
        self.n_training_steps = n_training_steps
        self.n_warmup_steps = n_warmup_steps
        self.criterion = nn.BCELoss()
        self.LABEL_COLUMNS = labels

        # Initialize F1 score metric for multi-label classification
        self.train_f1 = F1Score(task="multilabel", num_labels=n_classes, threshold=0.5)
        self.val_f1 = F1Score(task="multilabel", num_labels=n_classes, threshold=0.5)

    def forward(self, input_ids, attention_mask, labels=None):
        output = self.bert(input_ids, attention_mask=attention_mask)
        output = self.classifier(output.pooler_output)
        output = torch.sigmoid(output)
        loss = 0
        if labels is not None:
            loss = self.criterion(output, labels)
        return loss, output

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        loss, outputs = self(input_ids, attention_mask, labels)
        self.log("train_loss", loss, prog_bar=True, logger=True, on_step=True, on_epoch=True)

        # Update F1 score
        self.train_f1(outputs, labels)

        # Log F1 score on each step
        self.log("train_f1", self.train_f1, prog_bar=True, logger=True, on_step=True, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        loss, outputs = self(input_ids, attention_mask, labels)
        self.log("val_loss", loss, prog_bar=True, logger=True, on_step=True, on_epoch=True)

        # Update F1 score
        self.val_f1(outputs, labels)

        # Log F1 score on each step
        self.log("val_f1", self.val_f1, prog_bar=True, logger=True, on_step=True, on_epoch=True)

        return loss

    def on_train_epoch_end(self):
        # Reset F1 score at the end of the epoch
        self.train_f1.reset()

    def on_validation_epoch_end(self):
        # Reset F1 score at the end of the epoch
        self.val_f1.reset()

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=2e-5)
        scheduler = get_linear_schedule_with_warmup(
          optimizer,
          num_warmup_steps=self.n_warmup_steps,
          num_training_steps=self.n_training_steps
        )
        return dict(
          optimizer=optimizer,
          lr_scheduler=dict(
            scheduler=scheduler,
            interval='step'
          )
        )

In [ ]:
df_train = pd.read_csv(r"/content/drive/MyDrive/goemotions3.csv")
df_val = pd.read_csv(r"/content/drive/MyDrive/goemotions3.csv")
BERT_MODEL_NAME = 'bert-base-cased'
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_TOKEN_COUNT = 128
label_list = list(df_train.columns[9:])


N_EPOCHS = 15
BATCH_SIZE = 256
data_module = CustomDataModule(
    preprocessing(df_train),
    preprocessing(df_val),
    tokenizer,
    label_list,
    batch_size=BATCH_SIZE,
    max_token_len=MAX_TOKEN_COUNT
)

steps_per_epoch = len(df_train) // BATCH_SIZE
total_training_steps = steps_per_epoch * N_EPOCHS
warmup_steps = total_training_steps // 5
model = CustomBertModel(
    n_classes=len(label_list),
    labels=label_list,
    n_warmup_steps=warmup_steps,
    n_training_steps=total_training_steps
)

checkpoint_callback = ModelCheckpoint(
  dirpath='/content/drive/MyDrive/checkpoints',
  filename='QTag-{epoch:02d}-{val_loss:.2f}',
  save_top_k=1,
  verbose=True,
  monitor="val_loss",
  mode="min"
)

logger = TensorBoardLogger("lightning_logs", name="bert-sentiment")
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=2)
trainer = pl.Trainer(
    logger=logger,
    callbacks=[early_stopping_callback, checkpoint_callback],
    max_epochs=N_EPOCHS,
    accelerator='gpu',
    devices=1,
)

trainer.fit(model, data_module)


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.10/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/checkpoints exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: The ``compute`` method of metric MultilabelF1Score was called before the ``update`` method which may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)  # noqa: B028


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.utilities.rank_zero:Epoch 12, global step 3562: 'val_loss' reached 0.08237 (best 0.08237), saving model to '/content/drive/MyDrive/checkpoints/QTag-epoch=12-val_loss=0.08.ckpt' as top 1


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:Epoch 13, global step 3836: 'val_loss' reached 0.08042 (best 0.08042), saving model to '/content/drive/MyDrive/checkpoints/QTag-epoch=13-val_loss=0.08.ckpt' as top 1


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:Epoch 14, global step 4110: 'val_loss' reached 0.07949 (best 0.07949), saving model to '/content/drive/MyDrive/checkpoints/QTag-epoch=14-val_loss=0.08.ckpt' as top 1
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=15` reached.
